# Week 6: Regression modeling and inference

Many times we want to understand the relationship between two or more variables, or features, of our data. Previously, we learned how to plot variables against each other and analyze the figures descriptively. Today, we are going to work through a quantitative method of measuring relationships: regression models. 

You will need to download `ny_clean.csv` from Canvas under Files >> Data. 

This week's lesson is a simplied version of:  
- The [Week 8 on Linear Regression from General Assembly's Data Science Course](https://github.com/justmarkham/DAT4)


In [ ]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.formula.api as smf
import seaborn as sns

## Recap - Linear Models

Let's pull back in our `ny_clean.csv` data and rebuild our multiple linear model. 

In [ ]:
# read data into a DataFrame
ny_clean = pd.read_csv('ny_clean.csv')
ny_clean.head()

In [ ]:
# create a fitted model in one line, we did this last class
lm = smf.ols(formula='price ~ bedrooms + school_distance + livingArea + yearBuilt', data=ny_clean).fit()

# print the summary
lm.summary()

## 6.4 Feature Selection

How do I decide **which features to include** in a linear model? Here's one idea:
- Try different models, and only keep predictors in the model if they have small p-values.
- Check the collinearity between predictors
- Check whether the R-squared value goes up when you add new predictors.

What are the **drawbacks** to this approach?
- Linear models rely upon a lot of **assumptions** (such as the features being independent), and if those assumptions are violated (which they usually are), R-squared and p-values are less reliable.
- Using a p-value cutoff of 0.05 means that if you add 100 predictors to a model that are **pure noise**, 5 of them (on average) will still be counted as significant.
- R-squared is susceptible to **overfitting**, and thus there is no guarantee that a model with a high R-squared value will generalize. Below is an example:

First, let's check the collinearity between bedrooms, school distance, living area, and the year built. What would it mean for deciding on a strategy, if for instance, the correlation between living area and bedrooms were 1?

In [ ]:
ny_clean[['bedrooms', 'school_distance', 'livingArea', 'yearBuilt', 'price']].corr()

Since the correlation is pretty low here, we should not be too worried. As a rule of thumb, anything about |0.6| or above, I will do Variance Inflation Factor tests, where we find
$$
VIF = \frac{1}{1-R_i^2}
$$
where $i$ is the independent variable in question and we regress it on all the other independent variables. A good rule of thumb is to be concerned that multicollinearity exists if the Variance Inflation Factor (VIF) $\frac{1}{1-R^2}$ is 3 or above. 

In [ ]:
lm_vif=smf.ols(formula='livingArea ~ bedrooms + school_distance + yearBuilt', data=ny_clean).fit()
print("VIF for livingArea is ", 1/(1-lm_vif.rsquared))

## 6.5 Handling Categorical Predictors

### Two Categories

Up to now, all of our predictors have been numeric. What if one of our predictors was categorical?

Let's create a new feature called **historic** for any listing in a building older than 1925. 

In [ ]:
ny_clean['historic'] = ny_clean['yearBuilt'].apply(lambda x: 'historic' if x<=1925 else 'new')
ny_clean.head()

In [ ]:
# What does it look like as a distribution?
sns.displot(ny_clean, 
        x="price", 
        hue="historic",
        kind="kde", 
        fill=True,
        ).set(title = "Price by Historic Status")

In [ ]:
lm_cat = smf.ols(formula='price ~ bedrooms + school_distance + livingArea + historic', data=ny_clean).fit()
lm_cat.summary()

In [ ]:
## YOUR TURN 
## add in the variable describing if the school is public or charter (school_type)
## Run a multiple linear regression and print the summary


### More than Two Categories

In [ ]:
lm_cat = smf.ols(formula='price ~ bedrooms + school_distance + livingArea + historic + BoroName', data=ny_clean).fit()
lm_cat.summary()

What is happening under the hood? We have to represent BoroName numerically, but we can't simply code it as 0=Manhattan, 1=Bronx, 2=StatenIsland, etc. because that would imply an **ordered relationship** between boroughs (and thus Staten Island is somehow "twice" the Manhattan category).

Instead, we create **another dummy variable**:

In [ ]:
# create three dummy variables using get_dummies, then exclude the first dummy column
boro_dummies = pd.get_dummies(ny_clean['BoroName'], prefix='Boro').iloc[:, 1:]
boro_dummies.head()

In [ ]:
# change to 0s and 1s
boro_dummies = boro_dummies.astype(int)
boro_dummies.head()

In [ ]:
# concatenate the dummy variable columns onto the original DataFrame (axis=0 means rows, axis=1 means columns)
ny_clean = pd.concat([ny_clean, boro_dummies], axis=1)
ny_clean.head()

Here is how we interpret the coding:
- **Bronx** is coded as Boro_*=0
- **Manhattan** is coded as Boro_Manhattan=1 and Boro_*=0
- **Queens** is coded as Boro_Queens=1 and Boro_*=0

Why do we only need **four dummy variables, not five?** Because four dummies captures all of the information about the Borough feature, and implicitly defines the Bronx as the baseline level. (In general, if you have a categorical feature with k levels, you create k-1 dummy variables.)

If this is confusing, think about why we only needed one dummy variable for historic, not two dummy variables (historic and new).

Let's include the new dummy variables in the model separately:

In [ ]:
lm_cat = smf.ols(formula='price ~ bedrooms + school_distance + livingArea + historic + \
                Boro_Brooklyn + Boro_Manhattan + Boro_Queens + Boro_StatenIsland', data=ny_clean).fit()
lm_cat.summary()

How do we interpret the coefficients?
- Holding all other variables fixed, having a listing in **Manhattan** is associated with an average **increase** in price of \$1,173,000  (as compared to the baseline level, which is the Bronx).
- Being having a listing in **Staten Island** area is associated with an average **increase** in price of \$88,830 (as compared to the Bronx).

**A final note about dummy encoding:** If you have categories that can be ranked (i.e., strongly disagree, disagree, neutral, agree, strongly agree), you can potentially use a single dummy variable and represent the categories numerically (such as 1, 2, 3, 4, 5).

## 6.6 Transforming Variables

### Log Values

When our data is skewed or non-linear, we may wish to log-transform the variable. We often see this with price or income data, where the data may have large outliers or a non-linear distribution of values. 

If we log-transform a dependent (Y) value, we interpret the coefficient as: 

- $\log(y) = \beta_0 + \beta_1x_1 \rightarrow$ a unit change in X leads to a percent change in Y
- Example: coefficient is 0.198, $\exp(0.198)=1.22$. For every one-unit increase in X, Y increases by 22\%. 

If we log-transform an independent (X) value, we interpret the coefficient as: 

- $y = \beta_0 + \beta_1 \log(x_1) \rightarrow$ a percent change in X leads to a unit change in Y
- Example: coefficient is 0.198. For a 10\% increase in X, Y increases by $0.198*log(1.10)=0.02$ units


If we log-transform both, we interpret the coefficient as: 
- $\log(y) = \beta_0 + \beta_1 \log(x_1) \rightarrow$ a percent change in X leads to a percent change in Y
- Example: coefficient is 0.198. For a 10\% increase in X, Y increases $((1.10^{0.198})-1)*100=3.7\%$


In [ ]:
# For example, if we want to look predict the square footage by bedrooms and price
lm_cat = smf.ols(formula='livingArea ~ bedrooms + np.log(price)', data=ny_clean).fit()
lm_cat.summary()

### Interaction Terms

Also, sometimes terms may interfere with eachother's ability to be independent. We use interaction terms then to describe when there may be an additional relationship. 

For example, consider you are looking at a model predicting someone's support for a policy and you have variables like income, education, political ideology (conservative, liberal), and political party affiliation (republican, democrat). Likely, someone's political ideology is related to their party affiliation, so these terms will interact. We can account for this with "interaction terms".

In [ ]:
lm_cat = smf.ols(formula='price ~ bedrooms * livingArea', data=ny_clean).fit()
lm_cat.summary()

In [ ]:
## YOUR TURN 
## Pull in another dataset we've used (bike share trips, ny census, la census)
## Run a multiple linear regression



In [ ]:
## YOUR TURN
## Describe in words what you see in the model